# 6.3 Knowledge Distillation for Serving: Lab

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.3_distillation/lab.ipynb)
[![Open In Molab](https://img.shields.io/badge/Open%20in-Molab-blue)](https://molab.marimo.io/github/harshuljain13/llm-inference-at-scale/blob/master/content/07_scaling/06.3_distillation/lab.ipynb)

This lab quantifies distillation tradeoffs: SwiftKV prefill savings, Caprese quality recovery curves, and Llamba cross-architecture speedups.

In [ ]:
# --- Setup: install dependencies via subprocess (Colab/Molab compatible) ---
import subprocess
import sys
# Install numpy for numerical modeling and matplotlib for charts
subprocess.check_call([sys.executable, '-m', 'pip', 'install', '-q', 'numpy', 'matplotlib'])


In [ ]:
# --- Core imports ---
import numpy as np  # array math for throughput/latency models
import matplotlib.pyplot as plt  # plotting distillation tradeoff charts


## Experiment 1: SwiftKV Prefill FLOPs Savings

SwiftKV skips KV computation in later layers during prefill. We compute the theoretical FLOP reduction as a function of skip fraction.

In [ ]:
# === PARAMETERS (change these to explore different model configs) ===
# Llama-3.1-70B configuration
NUM_LAYERS = 80  # total transformer layers in the model
HIDDEN_DIM = 8192  # hidden dimension (determines matmul size)
SEQ_LEN = 4096  # input tokens processed during prefill

# --- Baseline: compute full prefill cost ---
# Each layer does 3 matmuls (Q, K, V), each [seq, hidden] x [hidden, hidden]
# FLOPs per matmul = 2 * M * N * K (multiply-accumulate counted as 2 ops)
flops_per_layer = 3 * 2 * SEQ_LEN * HIDDEN_DIM * HIDDEN_DIM
# Total baseline FLOPs = per-layer cost times number of layers
full_prefill_flops = flops_per_layer * NUM_LAYERS

# --- Sweep: vary skip fraction from 0% (no skip) to 75% (aggressive) ---
skip_fractions = np.linspace(0, 0.75, 16)  # 16 evenly spaced points
speedups = []  # will hold the speedup at each fraction

# Iterate over each skip fraction to compute resulting speedup
for frac in skip_fractions:
    # Calculate how many layers skip KV computation
    skipped_layers = int(NUM_LAYERS * frac)
    # Remaining layers do full Q,K,V computation
    active_layers = NUM_LAYERS - skipped_layers
    # Skipped layers only compute Q projection (1 of 3 matmuls)
    skipped_flops = (flops_per_layer / 3) * skipped_layers
    # Active layers pay full cost
    active_flops = flops_per_layer * active_layers
    # Total optimized cost
    total_flops = active_flops + skipped_flops
    # Speedup = baseline / optimized (higher = better)
    speedups.append(full_prefill_flops / total_flops)

# --- Visualization: speedup curve ---
# --- Create the speedup visualization ---
plt.figure(figsize=(9, 5))
# Plot speedup as a function of skip percentage
plt.plot(skip_fractions * 100, speedups, 'o-', color='#2563eb', linewidth=2)
# Mark SwiftKV paper's operating point (50% skip = 2x speedup)
plt.axvline(x=50, color='#991b1b', linestyle='--', alpha=0.7, label='SwiftKV paper (50%)')
# Axis labels and formatting
plt.xlabel('Layers Skipped (%)')
plt.ylabel('Prefill Speedup (x)')
plt.title('SwiftKV: Prefill Throughput vs Layer Skip Fraction')
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Print key result (should match paper's 2x claim)
print(f'At 50% skip: {speedups[8]:.2f}x prefill speedup (paper reports ~2x)')


## Experiment 2: Caprese Quality Recovery Curve

After aggressive pruning drops benchmark scores, Caprese uses LoRA distillation to recover quality. We model recovery as a function of distillation compute.

In [ ]:
# === PARAMETERS ===
# Teacher model's benchmark score (what we're trying to approach)
TEACHER_QUALITY = 82.0
# Distillation training budget in billions of tokens
TRAINING_TOKENS_B = np.array([0, 5, 10, 25, 50, 100, 200, 500])

# --- Compression scenarios with different starting points and recovery rates ---
compressions = {
    # 2x: mild pruning, starts at 72%, recovers quickly
    '2x compression': {'initial': 72.0, 'rate': 0.025},
    # 3x: moderate pruning, bigger quality gap to close
    '3x compression': {'initial': 65.0, 'rate': 0.018},
    # 4x: aggressive pruning, slow and expensive recovery
    '4x compression': {'initial': 58.0, 'rate': 0.012},
}

# --- Plot recovery curves ---
plt.figure(figsize=(9, 5))
# One color per compression level for visual clarity
colors = ['#2563eb', '#166534', '#991b1b']

# Plot each compression level as a separate curve
for (label, params), color in zip(compressions.items(), colors):
    # Exponential recovery: approaches teacher asymptotically
    gap = TEACHER_QUALITY - params['initial']  # max recoverable quality
    # Formula: initial + gap * (1 - e^(-rate * tokens))
    quality = params['initial'] + gap * (1 - np.exp(-params['rate'] * TRAINING_TOKENS_B))
    # Plot this compression level's recovery trajectory
    plt.plot(TRAINING_TOKENS_B, quality, 'o-', color=color, linewidth=2, label=label)

# Reference lines: teacher ceiling and acceptable threshold
plt.axhline(y=TEACHER_QUALITY, color='black', linestyle='--', alpha=0.5, label='Teacher')
# 95% of teacher = acceptable quality for production
plt.axhline(y=TEACHER_QUALITY * 0.95, color='#64748b', linestyle=':', alpha=0.7, label='95% threshold')

# Formatting
plt.xlabel('Distillation Tokens (Billions)')
plt.ylabel('Benchmark Score')
plt.title('Caprese: Quality Recovery vs Training Compute')
plt.legend(loc='lower right')
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()


## Experiment 3: Llamba Latency Scaling

Transformers have O(n^2) attention cost. Mamba (Llamba's target architecture) scales linearly. We compare decode latency as sequence length grows.

In [ ]:
# === PARAMETERS ===
# Sequence lengths to compare (powers of 2 from 512 to 32K)
SEQ_LENGTHS = np.array([512, 1024, 2048, 4096, 8192, 16384, 32768])

# --- Latency models ---
# Transformer: base overhead + quadratic attention component
# The O(n^2) term dominates at longer sequences
transformer_ms = 5 + 0.00015 * SEQ_LENGTHS**2 / 1000
# Mamba: base overhead + linear recurrence (constant state, no KV cache)
# Scales gracefully regardless of sequence length
mamba_ms = 3 + 0.008 * SEQ_LENGTHS

# --- Compute the speedup ratio at each sequence length ---
# Values > 1 mean Mamba is faster
speedup_llamba = transformer_ms / mamba_ms

# --- Dual-panel visualization ---
# --- Create side-by-side panels for absolute and relative comparison ---
fig_4, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 5))

# Left: absolute latency comparison
# Shows where the quadratic curve diverges from linear
ax1.plot(SEQ_LENGTHS / 1000, transformer_ms, 's-', color='#991b1b', linewidth=2, label='Transformer')
ax1.plot(SEQ_LENGTHS / 1000, mamba_ms, 'o-', color='#166534', linewidth=2, label='Mamba (Llamba)')
ax1.set_xlabel('Sequence Length (K tokens)')
ax1.set_ylabel('Decode Latency (ms/token)')
ax1.set_title('Latency: Transformer vs Llamba')
ax1.legend()
ax1.grid(True, alpha=0.3)

# Right: speedup factor (how much faster Mamba is)
ax2.bar(range(len(SEQ_LENGTHS)), speedup_llamba, color='#2563eb', alpha=0.8)
# Label x-axis with human-readable sequence lengths
ax2.set_xticks(range(len(SEQ_LENGTHS)))
ax2.set_xticklabels([f'{s//1000}K' for s in SEQ_LENGTHS], rotation=45)
ax2.set_xlabel('Sequence Length')
ax2.set_ylabel('Speedup (x)')
ax2.set_title('Llamba Speedup Factor')
ax2.grid(True, alpha=0.3, axis='y')

plt.tight_layout()
plt.show()

# Print the key numbers
print(f'Speedup at 4K: {speedup_llamba[3]:.1f}x')
print(f'Speedup at 32K: {speedup_llamba[-1]:.1f}x')


## Experiment 4: Combined Technique Comparison

Different optimization techniques trade off latency, throughput, quality, and memory differently. This visualization shows where each technique excels relative to FP16 baseline.

In [ ]:
# --- Technique characteristics (all relative to FP16 baseline = 1.0) ---
# latency: lower is better (0.5 = 2x faster)
# throughput: higher is better (2.0 = 2x more tokens/s)
# quality: higher is better (0.95 = 5% quality drop)
# memory: lower is better (0.3 = 70% memory savings)
techniques = {
    'INT4 Quant':     {'latency': 0.55, 'throughput': 1.8, 'quality': 0.97, 'memory': 0.30},
    'SwiftKV':        {'latency': 0.60, 'throughput': 2.0, 'quality': 0.99, 'memory': 0.85},
    'Caprese':        {'latency': 0.84, 'throughput': 1.16, 'quality': 0.96, 'memory': 0.76},
    'Llamba':         {'latency': 0.35, 'throughput': 2.5, 'quality': 0.95, 'memory': 0.70},
    'Quant+Distill':  {'latency': 0.35, 'throughput': 2.5, 'quality': 0.93, 'memory': 0.25},
}

# --- Build grouped bar chart for multi-axis comparison ---
# Four metrics to compare across all techniques
metric_names = ['latency', 'throughput', 'quality', 'memory']
# Human-readable axis labels with interpretation hint
metric_labels = ['Latency\n(lower=better)', 'Throughput\n(higher=better)',
                  'Quality\n(higher=better)', 'Memory\n(lower=better)']
# X positions for each metric group
x_positions = np.arange(len(metric_names))
# Width of each individual bar
bar_width = 0.15
# Distinct color per technique
colors = ['#2563eb', '#166534', '#991b1b', '#7c3aed', '#d97706']

# --- Render the grouped bar chart ---
fig_5, ax_5 = plt.subplots(figsize=(11, 6))
# Loop through techniques, placing bars side-by-side
for i, (name, metrics) in enumerate(techniques.items()):
    # Extract metric values in consistent order
    values = [metrics[m] for m in metric_names]
    # Offset bars horizontally so they don't overlap
    offset = x_positions + i * bar_width
    ax_5.bar(offset, values, bar_width, label=name, color=colors[i], alpha=0.85)

# Draw FP16 baseline reference at 1.0
ax_5.axhline(y=1.0, color='black', linestyle='--', alpha=0.4, label='Baseline')
# Center x-tick labels under each metric group
ax_5.set_xticks(x_positions + bar_width * 2)
ax_5.set_xticklabels(metric_labels)
ax_5.set_ylabel('Relative to FP16 Baseline')
ax_5.set_title('Distillation Techniques: Multi-Dimensional Comparison')
ax_5.legend(loc='upper right', fontsize=9)
ax_5.grid(True, alpha=0.2, axis='y')
plt.tight_layout()
plt.show()


## Key Takeaways

| Technique | Best For | Key Limitation |
|-----------|----------|----------------|
| **SwiftKV** | Prefill-bound workloads (long prompts) | No decode speedup |
| **Caprese** | Recovering quality after aggressive pruning | Needs 8+ GPU-hours |
| **Llamba** | Long-sequence inference (>4K tokens) | Architecture change required |
| **Quant + Distill** | Maximum compression | Compound quality risk |

**Recommendation**: Start with quantization (free). Add SwiftKV for prefill bottlenecks. Use Llamba only for dedicated long-context serving where KV cache is the binding constraint.